In [ ]:
# ============================================================
# 🧩 STEP 1: Environment Setup (T4-friendly)
# ============================================================
!pip install -q transformers accelerate datasets evaluate rouge-score bert-score bitsandbytes torch torchvision torchaudio

from google.colab import userdata
from huggingface_hub import login

# 🔐 Login to Hugging Face
login(token=userdata.get("HF_Token"), add_to_git_credential=True)
login(new_session=False)

import os, gc, re, time, torch, pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Device:", device)

# ============================================================
# 🧩 STEP 2: Load CommonsenseQA dataset (first 100 samples)
# ============================================================
dataset = load_dataset("tau/commonsense_qa")
test_data = dataset["validation"].select(range(100))
print(f"✅ Number of samples: {len(test_data)}")
print("✅ Columns:", test_data.column_names)

# ============================================================
# 🧩 STEP 3: Helper Functions
# ============================================================
def extract_option_answer(text):
    """Extract letter (A–E) from the model output."""
    match = re.search(r"\b([a-eA-E])\b", text)
    if match:
        return match.group(1).lower()
    return text.strip().lower()

def calculate_accuracy(preds, refs):
    correct = sum(p == r for p, r in zip(preds, refs))
    return correct / len(preds)

def compute_metrics(preds, refs):
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")

    rouge_scores = rouge.compute(predictions=preds, references=refs)
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="en")

    results = {f"ROUGE_{k}": v for k, v in rouge_scores.items()}
    results["BERTScore_F1"] = sum(bert_scores["f1"]) / len(bert_scores["f1"])
    results["Exact_Match"] = calculate_accuracy(preds, refs)
    results["Accuracy"] = results["Exact_Match"]
    return results

# ============================================================
# 🧩 STEP 4: Batched inference for Qwen2.5 (T4-safe)
# ============================================================
def run_inference_batched(model_name, dataset, batch_size=2, quantize_4bit=True):
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n🚀 Running inference for: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True,
    }

    # 🧠 Enable 4-bit quantization for T4 safety
    if quantize_4bit:
        model_kwargs["load_in_4bit"] = True

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    prompt_prefix = (
        "Choose the correct answer (a/b/c/d/e) without explanation.\n"
        "Question: "
    )
    inputs = [
        f"{prompt_prefix}{item['question']}\nOptions: {', '.join(item['choices']['text'])}\nAnswer:"
        for item in dataset
    ]
    refs = [item["answerKey"].lower() for item in dataset]
    preds = []

    start_time = time.time()
    for i in tqdm(range(0, len(inputs), batch_size), desc="Generating answers (batched)"):
        batch_inputs = inputs[i:i+batch_size]
        tokenized = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**tokenized, max_new_tokens=32)
        for out in outputs:
            pred = tokenizer.decode(out, skip_special_tokens=True)
            preds.append(extract_option_answer(pred))

    inference_time = time.time() - start_time
    results = compute_metrics(preds, refs)
    results["Inference_Time(s)"] = round(inference_time, 2)
    results["Model_Size(GB)"] = round(sum(p.numel() for p in model.parameters()) * 2 / (1024**3), 2)
    results["Peak_GPU_Memory(GB)"] = round(torch.cuda.max_memory_allocated(device) / (1024**3), 2)

    df_model = pd.DataFrame([results])
    file_name = f"{model_name.replace('/', '_')}_commonsenseqa_results_100.csv"
    df_model.to_csv(file_name, index=False)
    print(f"✅ Saved results for {model_name} to {file_name}")

    del model, tokenized, outputs
    torch.cuda.empty_cache()
    gc.collect()
    return results

# ============================================================
# 🧩 STEP 5: Run Qwen2.5 models sequentially (Colab T4)
# ============================================================
qwen_models = [
    ("Qwen/Qwen2.5-1.5B-Instruct", 2),
    ("Qwen/Qwen2.5-7B-Instruct", 1)
]

qwen_results = {}
for model_name, batch_size in qwen_models:
    qwen_results[model_name] = run_inference_batched(model_name, test_data, batch_size=batch_size)

# ============================================================
# 🧩 STEP 6: Display Results
# ============================================================
df_qwen = pd.DataFrame(qwen_results).T
print("\n📊 Baseline Vanilla Inference Results (100 samples, CommonsenseQA) for Qwen2.5 models:")
display(df_qwen)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 16.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Device: cuda


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generating answers (batched): 100%|██████████| 100/100 [05:34<00:00,  3.35s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Saved results for Qwen/Qwen2.5-7B-Instruct to Qwen_Qwen2.5-7B-Instruct_commonsenseqa_results_100.csv

📊 Baseline Vanilla Inference Results (100 samples, CommonsenseQA) for Qwen2.5 models:


,ROUGE_rouge1,ROUGE_rouge2,ROUGE_rougeL,ROUGE_rougeLsum,BERTScore_F1,Exact_Match,Accuracy,Inference_Time(s),Model_Size(GB),Peak_GPU_Memory(GB)
Qwen/Qwen2.5-1.5B-Instruct,0.17,0.0,0.17,0.17,0.957603,0.17,0.17,169.88,1.66,2.14
Qwen/Qwen2.5-7B-Instruct,0.17,0.0,0.17,0.17,0.957603,0.17,0.17,334.67,8.11,7.46
